# 0. Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
from geopy.distance import distance
from shapely.geometry import LineString, Point
from pyproj import Transformer
import ast

from sklearn.cluster import DBSCAN

In [30]:
MAX_DISTANCE = 2 # miles
MAX_TIME_DIFF = 30 # minutes

TRIPS_PATH = 'trips.csv'
USERS_PATH = 'users.csv'
FRIENDS_PATH = 'friendships.csv'

# 1. Load Data

In [4]:
trips = pd.read_csv(TRIPS_PATH)
users = pd.read_csv(USERS_PATH)
friendships = pd.read_csv(FRIENDS_PATH)

print(f'Loaded {len(trips):,} trips across {trips["user_id"].nunique()} users')
print(f'Loaded {len(users)} user profiles')
print(f'Loaded {len(friendships)} friendship edges')

Loaded 2,181 trips across 35 users
Loaded 35 user profiles
Loaded 69 friendship edges


# 2. Friendship Parsing

In [5]:
def parse_friendships(friendships):
    bidirectional = pd.concat([
        friendships.rename(columns={'user_id_1': 'user', 'user_id_2': 'friend'}),
        friendships.rename(columns={'user_id_2': 'user', 'user_id_1': 'friend'})
    ])

    friendship_dict = bidirectional.groupby('user')['friend'].apply(set).to_dict()
    return friendship_dict

In [6]:
friendships_dict = parse_friendships(friendships)
friendships_dict

{0: {17, 25, 27, 32},
 1: {7, 9, 18, 19, 21, 30},
 2: {24},
 3: {17, 20, 26, 33},
 4: {15, 28},
 5: {18},
 6: {11, 14, 16, 34},
 7: {1, 8, 9, 28, 29},
 8: {7, 18, 26, 30, 31},
 9: {1, 7, 12, 13, 14, 17, 28},
 10: {23, 26, 27, 33},
 11: {6, 20, 25, 33},
 12: {9, 25, 26},
 13: {9, 14, 17, 28},
 14: {6, 9, 13, 17, 18, 26, 28},
 15: {4, 25},
 16: {6, 18, 25, 33},
 17: {0, 3, 9, 13, 14, 22, 28},
 18: {1, 5, 8, 14, 16, 19, 31},
 19: {1, 18, 23, 28},
 20: {3, 11},
 21: {1, 23, 27, 32},
 22: {17, 29},
 23: {10, 19, 21, 24},
 24: {2, 23},
 25: {0, 11, 12, 15, 16, 26, 27},
 26: {3, 8, 10, 12, 14, 25, 30},
 27: {0, 10, 21, 25},
 28: {4, 7, 9, 13, 14, 17, 19},
 29: {7, 22},
 30: {1, 8, 26},
 31: {8, 18},
 32: {0, 21},
 33: {3, 10, 11, 16},
 34: {6}}

# 3. Identify Recurring Trips

In [14]:
def get_location_clusters(trips, buffer_miles: int = 1, min_samples: int = 2) -> pd.DataFrame:
    trips = trips.copy()
    for col, lat_col, lon_col in [
        ('start_cluster', 'start_lat', 'start_lon'),
        ('end_cluster', 'end_lat', 'end_lon')
    ]:
        coords = trips[[lat_col, lon_col]].values
        coords_rad = np.radians(coords)
        
        # eps in radians: buffer_miles / earth_radius
        eps = buffer_miles / 3958.8
        
        labels = DBSCAN(eps=eps, min_samples=min_samples, algorithm='ball_tree', metric='haversine').fit_predict(coords_rad)
        trips[col] = labels
    
    return trips

def get_recurring_trips(trips, min_occurrences=2):
    trips = get_location_clusters(trips)

    # drop trips where either endpoint was an outlier
    trips = trips[
        (trips['start_cluster'] != -1) &
        (trips['end_cluster'] != -1)
    ]

    # count occurrences of each OD cluster pair
    od_counts = (
        trips
        .groupby(['start_cluster', 'end_cluster'])
        .size()
        .reset_index(name='count')
    )

    recurring_od = od_counts[od_counts['count'] >= min_occurrences][['start_cluster', 'end_cluster']]

    return trips.merge(recurring_od, on=['start_cluster', 'end_cluster'], how='inner')

In [15]:
my_trips = trips[trips['user_id'] == 0]
recurring_trips = get_recurring_trips(my_trips)
recurring_trips

,trip_id,user_id,start_time,end_time,start_lat,start_lon,end_lat,end_lon,distance_miles,purpose,route_points,start_cluster,end_cluster
0,0,0,2025-01-01T07:57:00,2025-01-01T08:16:21.102077,33.895952,-84.463586,33.786555,-84.388670,11.304403,commute,"[(33.895788177856716, -84.4631460406156), (33....",0,0
1,1,0,2025-01-01T16:52:00,2025-01-01T17:19:39.716814,33.786555,-84.388670,33.895952,-84.463586,11.304403,commute,"[(33.78562842506307, -84.38849743080699), (33....",1,1
2,80,0,2025-01-02T07:04:00,2025-01-02T07:24:04.771227,33.895952,-84.463586,33.786555,-84.388670,11.304403,commute,"[(33.89534472107771, -84.46363333155884), (33....",0,0
3,161,0,2025-01-03T07:24:00,2025-01-03T07:58:05.982561,33.895952,-84.463586,33.786555,-84.388670,11.304403,commute,"[(33.896212068801965, -84.46354696964562), (33...",0,0
4,162,0,2025-01-03T18:11:00,2025-01-03T18:45:03.387092,33.786555,-84.388670,33.895952,-84.463586,11.304403,commute,"[(33.785590183791776, -84.38797806731486), (33...",1,1
5,390,0,2025-01-06T07:27:00,2025-01-06T07:46:19.775632,33.895952,-84.463586,33.786555,-84.388670,11.304403,commute,"[(33.894994766727805, -84.46350344953042), (33...",0,0
6,472,0,2025-01-07T07:19:00,2025-01-07T07:47:31.753348,33.895952,-84.463586,33.786555,-84.388670,11.304403,commute,"[(33.89643452390953, -84.46268806105337), (33....",0,0
7,473,0,2025-01-07T18:18:00,2025-01-07T18:49:01.076229,33.786555,-84.388670,33.895952,-84.463586,11.304403,commute,"[(33.78715853731851, -84.38805668804916), (33....",1,1
8,552,0,2025-01-08T07:35:00,2025-01-08T08:03:48.141874,33.895952,-84.463586,33.786555,-84.388670,11.304403,commute,"[(33.89554824114312, -84.46336239603058), (33....",0,0
9,553,0,2025-01-08T17:58:00,2025-01-08T18:27:49.436393,33.786555,-84.388670,33.895952,-84.463586,11.304403,commute,"[(33.78618342209218, -84.38892193860988), (33....",1,1


# 4. Score Trips

In [32]:
def geodesic(a, b):
    dist_miles = distance(a, b).miles
    return dist_miles

def preprocess_trips(trips):
    #type conversion
    trips = trips.copy()
    trips['start_time'] = pd.to_datetime(trips['start_time'], format='mixed')
    trips['end_time'] = pd.to_datetime(trips['end_time'], format='mixed')
    if isinstance(trips['route_points'].iloc[0], str):
        trips['route_points'] = trips['route_points'].apply(ast.literal_eval)
    
    # feature engineering for recurring trip detection
    trips['time_bucket'] = trips['start_time'].dt.floor('30min').dt.time
    trips['day_of_week'] = trips['start_time'].dt.dayofweek
    return trips

def get_utm_transformer(lat, lon):
    # get the appropriate UTM zone transformer for a given location
    utm_zone = int((lon + 180) / 6) + 1
    hemisphere = "north" if lat >= 0 else "south"
    epsg = 32600 + utm_zone if hemisphere == "north" else 32700 + utm_zone
    return Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)

def project_and_create_polyline(route_points, transformer):
    route_proj = [transformer.transform(lon, lat) for lat, lon in route_points]
    return LineString(route_proj)

def get_pickup_begin_score(my_trip, other_trip, max_distance):
    # how close other trip's start is to my trip's start
    my_start = (my_trip['start_lat'], my_trip['start_lon'])
    other_start = (other_trip['start_lat'], other_trip['start_lon'])
    distance = geodesic(my_start, other_start)
    pickup_score = 1 - min(distance / max_distance, 1)
    return pickup_score

def get_pickup_enroute_score(my_route, other_trip, transformer, max_distance):
    # how close other trip's start is to any of my trip's route points
    METERS_TO_MILES = 0.000621371
    other_start = Point(transformer.transform(other_trip['start_lon'], other_trip['start_lat']))
    distance = other_start.distance(my_route) * METERS_TO_MILES
    pickup_score = 1 - min(distance / max_distance, 1)
    return pickup_score

def get_dropoff_end_score(my_trip, other_trip, max_distance):
    # how close other trip's end is to my trip's end
    my_end = (my_trip['end_lat'], my_trip['end_lon'])
    other_end = (other_trip['end_lat'], other_trip['end_lon'])
    distance = geodesic(my_end, other_end)
    dropoff_score = 1 - min(distance / max_distance, 1)
    return dropoff_score

def get_dropoff_enroute_score(my_route, other_trip, transformer, max_distance):
    # how close other trip's end is to any of my trip's route points
    METERS_TO_MILES = 0.000621371
    other_end = Point(transformer.transform(other_trip['end_lon'], other_trip['end_lat']))
    distance = other_end.distance(my_route) * METERS_TO_MILES
    dropoff_score = 1 - min(distance / max_distance, 1)
    return dropoff_score

def get_start_time_score(my_trip, other_trip, max_time_diff):
    time_diff = abs(my_trip['start_time'] - other_trip['start_time']).total_seconds() / 60
    time_score = 1 - min(time_diff / max_time_diff, 1)
    return time_score

def get_end_time_score(my_trip, other_trip, max_time_diff):
    time_diff = abs(my_trip['end_time'] - other_trip['end_time']).total_seconds() / 60
    time_score = 1 - min(time_diff / max_time_diff, 1)
    return time_score


def score_trips_for_user(user_id, trips, max_distance, max_time_diff, test=False, verbose=False):
    trips = preprocess_trips(trips)
    my_trips = trips[trips['user_id'] == user_id]
    my_recurring_trips = get_recurring_trips(my_trips)

    if my_recurring_trips.empty:
        print("No recurring trips found for user", user_id)
        return []

    my_friends = parse_friendships(friendships).get(user_id, set())
    friend_trips = [get_recurring_trips(trips[trips['user_id'] == friend]) for friend in my_friends]
    friend_trips = [df for df in friend_trips if not df.empty]
    other_recurring_trips = pd.concat(friend_trips) if friend_trips else pd.DataFrame()

    if other_recurring_trips.empty:
        print("No recurring trips found for friends of user", user_id)
        return []

    scores = []
    for _, my_trip in my_recurring_trips.iterrows():
        if verbose:
            print("Scoring trip:", my_trip['trip_id'])
            
        transformer = get_utm_transformer(my_trip['start_lat'], my_trip['start_lon'])
        my_route = project_and_create_polyline(my_trip['route_points'], transformer)

        for i, other_trip in other_recurring_trips.iterrows():
            pickup_begin_score = get_pickup_begin_score(my_trip, other_trip, max_distance)
            dropoff_end_score = get_dropoff_end_score(my_trip, other_trip, max_distance)
            
            pickup_enroute_score = get_pickup_enroute_score(my_route, other_trip, transformer, max_distance)
            if pickup_begin_score == 0 and pickup_enroute_score == 0:
                # no good pickup options, so skip this trip
                continue
            dropoff_enroute_score = get_dropoff_enroute_score(my_route, other_trip, transformer, max_distance)
            if dropoff_end_score == 0 and dropoff_enroute_score == 0:
                # no good dropoff options, so skip this trip
                continue

            start_time_score = get_start_time_score(my_trip, other_trip, max_time_diff)
            end_time_score = get_end_time_score(my_trip, other_trip, max_time_diff)

            # can pickup and dropoff at end, or pickup enroute and dropoff at end, or pickup at beginning and dropoff enroute, or pickup and dropoff enroute
            # time alignment should match with pickup/dropoff alignment - if pickup is enroute, then start time should be less important, if dropoff is enroute, then end time should be less important
            route_score = max(pickup_begin_score + dropoff_end_score + 0.5*start_time_score + 0.5*end_time_score,
                              pickup_enroute_score + dropoff_end_score + 0.25*start_time_score + 0.75*end_time_score,
                              pickup_begin_score + dropoff_enroute_score + 0.75*start_time_score + 0.25*end_time_score,
                              pickup_enroute_score + dropoff_enroute_score + 0.5*start_time_score + 0.5*end_time_score)

            
            total_score = route_score / 3
            scores.append({
                'user_id': user_id,
                'other_user': other_trip['user_id'],
                'my_trip': my_trip['trip_id'],
                'other_trip': other_trip['trip_id'],
                'score': round(total_score, 6)
            })

        if test:
            break
    
    scores.sort(key=lambda x: x['score'], reverse=True)
    return scores

In [33]:
# runtime ~12s per user
scores = score_trips_for_user(0, trips, max_distance=MAX_DISTANCE, max_time_diff=MAX_TIME_DIFF, test=False)

In [35]:
all_scores = {}
for user_id in trips['user_id'].unique():
    user_id = int(user_id)
    print(f"Scoring trips for user {user_id}...")
    all_scores[user_id] = score_trips_for_user(user_id, trips, max_distance=MAX_DISTANCE, max_time_diff=MAX_TIME_DIFF)

Scoring trips for user 0...
Scoring trips for user 1...
Scoring trips for user 2...
Scoring trips for user 3...
Scoring trips for user 4...
Scoring trips for user 5...
Scoring trips for user 6...
Scoring trips for user 7...
Scoring trips for user 8...
Scoring trips for user 9...
Scoring trips for user 10...
Scoring trips for user 11...
Scoring trips for user 12...
Scoring trips for user 13...
Scoring trips for user 14...
Scoring trips for user 15...
Scoring trips for user 16...
Scoring trips for user 17...
Scoring trips for user 18...
Scoring trips for user 19...
Scoring trips for user 20...
Scoring trips for user 21...
Scoring trips for user 22...
Scoring trips for user 23...
Scoring trips for user 24...
Scoring trips for user 25...
Scoring trips for user 26...
Scoring trips for user 27...
Scoring trips for user 28...
Scoring trips for user 29...
Scoring trips for user 30...
Scoring trips for user 31...
Scoring trips for user 32...
Scoring trips for user 33...
Scoring trips for user 3

In [36]:
all_matches = [s for scores in all_scores.values() for s in scores if s['score'] > 0]
len(all_matches)

33576

In [40]:
matches_df = pd.DataFrame(all_matches)
matches_df.sort_values('score', ascending=False, inplace=True)
matches_df

,user_id,other_user,my_trip,other_trip,score
23330,21,1,1699,1653,0.913032
5161,9,1,1205,1186,0.892545
23331,21,1,1233,1187,0.886815
5162,9,1,494,475,0.884913
5163,9,1,575,554,0.877086
...,...,...,...,...,...
3888,6,14,96,114,0.031955
3905,6,14,96,2049,0.031955
3904,6,14,96,1840,0.031955
3890,6,14,96,423,0.031955
